In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate fastdtw kmedoids scikit-learn pandas numpy matplotlib tqdm

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import string
import traceback
import copy
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Sequence, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

from transformers import AutoTokenizer, AutoModel

from fastdtw import fastdtw
from sklearn.ensemble import IsolationForest

import kmedoids as km

In [ ]:
"""Manages system and output path structures for text directories and pipeline progress.

    Attributes:
        project_root & results_root: Root directories for source inputs and generated artifacts.
        torah_dir & imposters_dir: Core directory paths for sub-corpora data mapping.
        progress_file: Path to the JSON state logger file.
    """

@dataclass
class PipelinePaths:
    project_root: Path
    results_root: Path

    torah_dir: Path = field(init=False)
    imposters_dir: Path = field(init=False)
    imposters1_dir: Path = field(init=False)
    progress_file: Path = field(init=False)

    def __post_init__(self):
        self.torah_dir = self.project_root / "torah_chapters"
        self.imposters_dir = self.project_root / "imposters"
        self.imposters1_dir = self.project_root / "imposters1"
        self.results_root.mkdir(parents=True, exist_ok=True)
        self.progress_file = self.results_root / "progress.json"

"""Holds hyperparameters for transformers, MC dropout sampling, training, and clustering.

    Attributes:
        bert_model_name: HuggingFace model identifier, defaulting to 'avichr/heBERT'.
        segment_word_count & batch_factor: Split constraints managing text segmentation size.
        mc_samples: Number of Monte Carlo dropout forward passes for uncertainty tracking.
        freeze_bert: Boolean flag enforcing parameter freezing on the underlying transformer base.
        device: Dynamically resolved compute target string ('cuda' or 'cpu').
    """
@dataclass
class ExperimentConfig:

    bert_model_name: str = "avichr/heBERT"
    max_length: int = 128
    segment_word_count: int = 50
    batch_factor: int = 2
    mc_samples: int = 15
    num_labels: int = 2
    epochs: int = 12

    batch_size: int = 8
    learning_rate: float = 2e-4
    dropout_rate: float = 0.2
    freeze_bert: bool = True

    validation_split: float = 0.25
    random_state: int = 0
    early_stopping_patience: int = 3
    best_model_metric: str = "val_loss"

    kmedoids_clusters: int = 2
    isolation_threshold: float = 0.00005 / 2
    summary_percentile: int = 90
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    signal_for_distance: str = "weighted"

"""Holds hyperparameters for transformers, MC dropout sampling, training, and clustering.

    Attributes:
        bert_model_name: HuggingFace model identifier, defaulting to 'avichr/heBERT'.
        segment_word_count & batch_factor: Split constraints managing text segmentation size.
        mc_samples: Number of Monte Carlo dropout forward passes for uncertainty tracking.
        freeze_bert: Boolean flag enforcing parameter freezing on the underlying transformer base.
        device: Dynamically resolved compute target string ('cuda' or 'cpu').
    """
@dataclass
class PipelineState:

    iteration_number: int = 1
    completed_pairs: List[Tuple[str, str]] = field(default_factory=list)
    database: Dict[str, List[str]] = field(default_factory=dict)

    forest_report: Dict[str, Sequence[float]] = field(default_factory=dict)
    summary_report: Dict[str, Sequence[float]] = field(default_factory=dict)
    distance_label_report: Dict[str, Sequence[int]] = field(default_factory=dict)


def set_global_seed(seed: int):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
"""Manages file storage indexing, directory queries, and disk-to-RAM corpus loading.

    Attributes:
        paths & state: Pipeline infrastructure resources for folder discovery and database populating.
        list_author_names_from_imposters / 1: Queries and returns sorted subfolder names for each imposter split.
        load_torah_corpus: Reads all Torah books, updates the state database, and returns a key-text mapping.
        _read_texts_as_map: Static helper that reads text files and matches their content to their filename stems.
    """

class CorpusRepository:

    def __init__(self, paths: PipelinePaths, state: PipelineState):
        self.paths = paths
        self.state = state

    def list_author_names_from_imposters(self) -> List[str]:

        return self._list_author_names(self.paths.imposters_dir)

    def list_author_names_from_imposters1(self) -> List[str]:

        return self._list_author_names(self.paths.imposters1_dir)

    def list_torah_file_stems(self) -> List[str]:

        return sorted([file.stem for file in self.paths.torah_dir.glob("*.txt")])

    def load_torah_corpus(self) -> Dict[str, str]:

        torah_map = self._read_texts_as_map(self.paths.torah_dir)
        self.state.database["torah"] = list(torah_map.values())
        return torah_map

    def load_imposter_group_by_name(self, author_name: str, group_name: str) -> List[str]:
        if group_name == "imposters":
            source_dir = self.paths.imposters_dir / author_name
        elif group_name == "imposters1":
            source_dir = self.paths.imposters1_dir / author_name
        else:
            raise ValueError("group_name must be 'imposters' or 'imposters1'")

        texts = self._read_all_texts(source_dir)
        self.state.database[author_name] = texts
        return texts

    @staticmethod
    def _list_author_names(folder: Path) -> List[str]:
        if not folder.exists():
            return []
        return sorted([item.name for item in folder.iterdir() if item.is_dir()])

    @staticmethod
    def _read_all_texts(folder: Path) -> List[str]:

        if not folder.exists():
            raise FileNotFoundError(f"Missing folder: {folder}")

        texts = []
        for file_path in sorted(folder.glob("*.txt")):
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                texts.append(f.read().replace("\n", " "))
        return texts

    @staticmethod
    def _read_texts_as_map(folder: Path) -> Dict[str, str]:

        if not folder.exists():
            raise FileNotFoundError(f"Missing folder: {folder}")

        result = {}
        for file_path in sorted(folder.glob("*.txt")):
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                result[file_path.stem] = f.read().replace("\n", " ")
        return result

In [ ]:
"""Handles text cleaning, diacritic stripping, and regex-based tokenization for Hebrew corpora.

    Attributes:
        punctuation_table: Translation lookup used to strip native and standard punctuation marks.
        clean_text: Normalizes string structures by filtering out non-Hebrew elements, newlines, and extra whitespace.
        preprocess_text: Transforms a raw text string into a filtered list of valid Hebrew-only tokens.
        preprocess_collection: Maps processing workflows across an entire array of corpus text inputs.
    """

class TextPreprocessor:
    def __init__(self):

        self.punctuation_table = str.maketrans("", "", string.punctuation + "״" + "׳")

    def remove_hebrew_diacritics(self, text: str) -> str:

        return re.sub(r"[\u0591-\u05C7]", "", text)

    def remove_english_letters(self, text: str) -> str:

        return re.sub(r"[A-Za-z]", "", text)

    def clean_text(self, text: str) -> str:

        text = text.replace("\n", " ")
        text = self.remove_hebrew_diacritics(text)
        text = self.remove_english_letters(text)
        text = text.translate(self.punctuation_table)
        text = re.sub(r"[^א-ת\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def preprocess_text(self, text: str) -> List[str]:

        cleaned = self.clean_text(text)
        tokens = cleaned.split()
        tokens = [token for token in tokens if re.fullmatch(r"[א-ת]+", token)]
        return tokens

    def preprocess_collection(self, texts: List[str]) -> List[List[str]]:

        return [self.preprocess_text(text) for text in texts]

In [ ]:

"""Chunks flat token lists into uniform, space-separated string segments of a fixed length.

    Attributes:
        segment_word_count: Default target number of tokens/words assigned to each segment block.
        tokens_to_segments: Slices a single token list into continuous, complete word chunks, dropping remainders.
        collection_to_segments: Flattens a whole collection of tokenized texts into a combined list of segments.
        collection_to_segments_by_text: Maps text segment arrays back to their corresponding file or document names.
    """
class Segmenter:

    def __init__(self, segment_word_count: int):
        self.segment_word_count = segment_word_count
    def tokens_to_segments(self, tokens: List[str], segment_word_count: Optional[int] = None) -> List[str]:
        size = segment_word_count or self.segment_word_count
        if len(tokens) < size:
            return []
        segments = []
        full_count = len(tokens) // size
        for i in range(full_count):
            start = i * size
            end = start + size
            segments.append(" ".join(tokens[start:end]))
        return segments
    def collection_to_segments(self, tokenized_texts: List[List[str]]) -> List[str]:
        all_segments = []
        for tokens in tokenized_texts:
            all_segments.extend(self.tokens_to_segments(tokens))
        return all_segments
    def collection_to_segments_by_text(self, tokenized_texts: List[List[str]], names: List[str]) -> Dict[str, List[str]]:
        if len(tokenized_texts) != len(names):
            raise ValueError("tokenized_texts and names must have the same length")
        return {name: self.tokens_to_segments(tokens) for name, tokens in zip(names, tokenized_texts)}

## Cell 7 — Sen2Pro-style BERT Encoder: Brief Mathematical Explanation

In this notebook, **Sen2Pro-style** refers strictly to one thing: capturing **model uncertainty** using **MC Dropout only**.

There is **no data augmentation** applied whatsoever:
* No word deletion
* No word substitution
* No word reordering
* No word insertion

For a single segment, BERT is executed multiple times with dropout enabled to obtain $N$ stochastic forward passes:

$$x_1, x_2, \ldots, x_N$$

Where each $x_i$ is an embedding vector of size `hidden_size` (e.g., 768).

We then compute the empirical mean vector:

$$\mu = \frac{1}{N}\sum_{i=1}^N x_i$$

And the variance for each individual dimension:

$$var_j = \frac{1}{N}\sum_{i=1}^N (x_{ij} - \mu_j)^2$$

This represents a **diagonal covariance** approach: instead of computing a full $768 \times 768$ covariance matrix, we only compute the variance per feature dimension.

The final representation passed to the classifier is the concatenation of the mean and variance vectors:

$$z = \text{concat}(\mu, var)$$

If `hidden_size = 768`, the resulting vector $z$ will have a dimensionality of 1536.

The overall uncertainty of the segment is defined as the mean of the feature variances:

$$u = \text{mean}(var)$$

Consequently, its weight during the style signal phase is determined via inverse uncertainty weighting:

$$w = \frac{1}{1+u}$$

In [ ]:
"""Wrapper for transformer-based embedding extraction leveraging Monte Carlo Dropout sampling.

    Attributes:
        tokenizer & model: Loaded HuggingFace transformer primitives optimized for Hebrew text.
        pooling: Aggregation strategy applied over the token dimension ('mean' or 'cls').
        encode_segments: Dispatches text blocks through MC loops to extract parameter means and variances.
        representation: The resulting feature vector created by concatenating mean ($mu$) and variance ($var$).
        uncertainty: Derived structural variance score computed by taking the row-wise mean of feature variance.
    """

class Sen2ProBertEncoder:

    def __init__(self, config: ExperimentConfig, pooling: str = "mean"):
        self.config = config
        self.pooling = pooling
        self.device = torch.device(config.device)

        print(f"Loading tokenizer/model: {config.bert_model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(config.bert_model_name)
        self.model = AutoModel.from_pretrained(config.bert_model_name)
        self.model.to(self.device)

        self.hidden_size = int(self.model.config.hidden_size)
        self.input_dim = self.hidden_size * 2

        if config.freeze_bert:
            print("BERT is frozen: requires_grad=False for all BERT parameters")
            for param in self.model.parameters():
                param.requires_grad = False

    def mean_pooling(self, last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:

        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def get_segment_embedding(self, outputs: Any, attention_mask: torch.Tensor) -> torch.Tensor:

        if self.pooling == "cls":
            return outputs.last_hidden_state[:, 0, :]
        if self.pooling == "mean":
            return self.mean_pooling(outputs.last_hidden_state, attention_mask)
        raise ValueError("pooling must be 'mean' or 'cls'")

    @torch.no_grad()
    def encode_segments(
        self,
        segments: List[str],
        batch_size: Optional[int] = None,
        return_mu_var: bool = False,
    ):
        batch_size = batch_size or self.config.batch_size

        if len(segments) == 0:
            empty_repr = torch.empty((0, self.input_dim), dtype=torch.float32)
            empty_unc = torch.empty((0,), dtype=torch.float32)
            if return_mu_var:
                return empty_repr, empty_unc, torch.empty((0, self.hidden_size)), torch.empty((0, self.hidden_size))
            return empty_repr, empty_unc
        self.model.train()

        all_representations = []
        all_uncertainties = []
        all_mu = []
        all_var = []

        for start in range(0, len(segments), batch_size):
            batch_segments = segments[start:start + batch_size]

            encoded = self.tokenizer(
                batch_segments,
                padding=True,
                truncation=True,
                max_length=self.config.max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}

            mc_embeddings = []
            for _ in range(self.config.mc_samples):
                outputs = self.model(**encoded)
                emb = self.get_segment_embedding(outputs, encoded["attention_mask"])
                mc_embeddings.append(emb.detach().cpu())
            mc_stack = torch.stack(mc_embeddings, dim=0)
            mu = mc_stack.mean(dim=0)
            var = mc_stack.var(dim=0, unbiased=False)

            representation = torch.cat([mu, var], dim=1)
            uncertainty = var.mean(dim=1)

            all_representations.append(representation)
            all_uncertainties.append(uncertainty)

            if return_mu_var:
                all_mu.append(mu)
                all_var.append(var)

        representations = torch.cat(all_representations, dim=0)
        uncertainties = torch.cat(all_uncertainties, dim=0)

        if return_mu_var:
            mu_vectors = torch.cat(all_mu, dim=0)
            var_vectors = torch.cat(all_var, dim=0)
            return representations, uncertainties, mu_vectors, var_vectors

        return representations, uncertainties

In [ ]:
"""PyTorch Dataset container wrapper for training features and target labels.

    Attributes:
        X: Feature tensor cast to floating-point representation.
        y: Target label tensor cast to long-integer format.
    """
class AuthorPairDataset(Dataset):

    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        self.X = X.float()
        self.y = y.long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

"""
Balances dataset representations by downsampling the majority class to match the minority class size.

Attributes:
    a_representations & b_representations: Extracted tensor embeddings for each author block.
    min_count: The exact target size derived from the smaller of the two input pools.
    perm: Random permutation indices used to shuffle combined tensors uniformly.
"""
def build_balanced_training_pairs(
    a_representations: torch.Tensor,
    b_representations: torch.Tensor,
    random_state: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    len_a = a_representations.shape[0]
    len_b = b_representations.shape[0]
    min_count = min(len_a, len_b)

    if min_count == 0:
        raise ValueError(f"Cannot build training set: len(A)={len_a}, len(B)={len_b}")

    rng = np.random.default_rng(random_state)
    idx_a = rng.choice(len_a, size=min_count, replace=False)
    idx_b = rng.choice(len_b, size=min_count, replace=False)

    X_a = a_representations[idx_a]
    X_b = b_representations[idx_b]

    y_a = torch.zeros(min_count, dtype=torch.long)
    y_b = torch.ones(min_count, dtype=torch.long)

    X = torch.cat([X_a, X_b], dim=0)
    y = torch.cat([y_a, y_b], dim=0)

    perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(random_state))
    X = X[perm]
    y = y[perm]

    return X, y

In [ ]:

"""Multi-Layer Perceptron (MLP) architecture used for author classification.

    This network acts as the prediction head, consuming concatenated transformer
    representations (mu and var) to separate and classify segment authors.

    Attributes:
        network: Sequential stack containing Linear, ReLU, Dropout, and output Linear layers.
        forward: Performs a standard forward propagation pass over the input tensor features.
    """
class Sen2ProAuthorClassifier(nn.Module):
    def __init__(self, input_dim: int, num_labels: int = 2, dropout_rate: float = 0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [ ]:
"""Handles deep learning classification lifecycle, training routines, and optimization.

    Coordinates data splitting, training loops, validation evaluation, and model checkpointing
    based on Early Stopping validation performance.

    Attributes:
        config: ExperimentConfig instance managing parameters like patience and learning rate.
        train_classifier: Prepares data, initializes optimization loops, and checkpoints the best parameters.
        _run_epoch: Internal runner managing single feedforward/backward iteration cycles across batches.
        history: Metrics store tracking loss, accuracy, and termination metadata per iteration.
    """

class Trainer:
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.device = torch.device(config.device)

    def train_classifier(self, X: torch.Tensor, y: torch.Tensor, input_dim: int):

        dataset = AuthorPairDataset(X, y)
        total_size = len(dataset)

        if total_size < 4:
            raise ValueError(
                f"Training set too small. Got only {total_size} samples."
            )

        val_size = max(1, int(total_size * self.config.validation_split))
        train_size = total_size - val_size

        if train_size <= 0:
            raise ValueError("Training set too small after validation split")

        if total_size < 40:
            print(
                "Warning: very small training set. "
                "Validation metrics may be unstable."
            )

        generator = torch.Generator().manual_seed(self.config.random_state)
        train_dataset, val_dataset = random_split(
            dataset,
            [train_size, val_size],
            generator=generator
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False
        )

        model = Sen2ProAuthorClassifier(
            input_dim=input_dim,
            num_labels=self.config.num_labels,
            dropout_rate=self.config.dropout_rate,
        ).to(self.device)

        criterion = nn.CrossEntropyLoss()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.config.learning_rate
        )

        history = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": [],
            "best_epoch": None,
            "best_val_loss": None,
            "stopped_early": False,
            "epochs_ran": 0,
        }

        best_val_loss = float("inf")
        best_state_dict = None
        epochs_without_improvement = 0

        for epoch in range(1, self.config.epochs + 1):
            train_loss, train_acc = self._run_epoch(
                model=model,
                loader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                train=True
            )

            val_loss, val_acc = self._run_epoch(
                model=model,
                loader=val_loader,
                criterion=criterion,
                optimizer=None,
                train=False
            )

            history["train_loss"].append(train_loss)
            history["train_accuracy"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_acc)
            history["epochs_ran"] = epoch

            print(
                f"Epoch {epoch}/{self.config.epochs} | "
                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
            )
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

                history["best_epoch"] = epoch
                history["best_val_loss"] = float(best_val_loss)

                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epochs_without_improvement >= self.config.early_stopping_patience:
                history["stopped_early"] = True

                print(
                    f"Early stopping: no validation loss improvement for "
                    f"{self.config.early_stopping_patience} epochs."
                )
                break

        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)
            model.to(self.device)

            print(
                f"Loaded best classifier from epoch "
                f"{history['best_epoch']} "
                f"with val_loss={history['best_val_loss']:.4f}"
            )

        return model, history

    def _run_epoch(self, model, loader, criterion, optimizer=None, train: bool = True):
        if train:
            model.train()
        else:
            model.eval()

        total_loss = 0.0
        total_correct = 0
        total_count = 0

        for batch_X, batch_y in loader:
            batch_X = batch_X.to(self.device)
            batch_y = batch_y.to(self.device)

            if train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(train):
                logits = model(batch_X)
                loss = criterion(logits, batch_y)

                if train:
                    loss.backward()
                    optimizer.step()

            preds = logits.argmax(dim=1)
            total_correct += (preds == batch_y).sum().item()
            total_loss += loss.item() * batch_X.size(0)
            total_count += batch_X.size(0)

        avg_loss = total_loss / max(total_count, 1)
        avg_acc = total_correct / max(total_count, 1)

        return avg_loss, avg_acc

In [ ]:

"""Infers author probabilities across Torah segments using the trained classifier and embedding uncertainty.

    Generates prediction probabilities alongside inverse-uncertainty weights used to filter or
    scale downstream distance matrix calculations.

    Attributes:
        config & encoder: Configurations and the underlying heBERT architecture used for segment encoding.
        predict_torah_segments: Evaluates segments by file, calculating author-B probabilities and inverse variance weights.
        weights: Quality metrics computed dynamically as $$1.0 / (1.0 + uncertainties\_np)$$.
    """
class TorahPredictor:
    def __init__(self, config: ExperimentConfig, encoder: Sen2ProBertEncoder):
        self.config = config
        self.encoder = encoder
        self.device = torch.device(config.device)

    @torch.no_grad()
    def predict_torah_segments(
        self,
        classifier: nn.Module,
        torah_segments_by_file: Dict[str, List[str]],
    ):
        classifier.eval()

        raw_prediction_map = {}
        uncertainty_map = {}
        weight_map = {}
        detailed_rows = []

        for file_name, segments in tqdm(torah_segments_by_file.items(), desc="Predicting Torah chapters"):
            if len(segments) == 0:
                raw_prediction_map[file_name] = np.array([], dtype=float)
                uncertainty_map[file_name] = np.array([], dtype=float)
                weight_map[file_name] = np.array([], dtype=float)
                continue

            representations, uncertainties = self.encoder.encode_segments(segments, batch_size=self.config.batch_size)
            representations = representations.float().to(self.device)

            probs_list = []
            for start in range(0, representations.shape[0], self.config.batch_size):
                batch_X = representations[start:start + self.config.batch_size]
                logits = classifier(batch_X)
                probabilities = torch.softmax(logits, dim=1)
                p_author_b = probabilities[:, 1]
                probs_list.append(p_author_b.detach().cpu())

            predictions = torch.cat(probs_list).numpy()
            uncertainties_np = uncertainties.numpy()
            weights = 1.0 / (1.0 + uncertainties_np)

            raw_prediction_map[file_name] = predictions
            uncertainty_map[file_name] = uncertainties_np
            weight_map[file_name] = weights

            for idx, (pred, unc, weight) in enumerate(zip(predictions, uncertainties_np, weights)):
                detailed_rows.append({
                    "file_name": file_name,
                    "segment_index": idx,
                    "prediction_P_B": float(pred),
                    "uncertainty": float(unc),
                    "weight": float(weight),
                })

            del representations, uncertainties
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return raw_prediction_map, uncertainty_map, weight_map, detailed_rows

## Cell 12 — Style Signal: Brief Mathematical Explanation

For each biblical chapter, we generate a sequence of segment-level predictions.

In this notebook, each prediction is consistently defined as:

$$p_i = P(\text{class}=1) = P(\text{Author B})$$

This ensures that the direction of the signal remains strictly consistent across all imposter pairs.

For a chunk of $k$ predictions (where $k = \text{batch\_factor}$):

$$p_1, p_2, \ldots, p_k$$

And their corresponding uncertainty-based weights:

$$w_1, w_2, \ldots, w_k$$

We compute two types of signals:

1. **Regular Signal** (Simple Arithmetic Mean):

$$\text{regular} = \frac{1}{k}\sum_{i=1}^k p_i$$

2. **Weighted Signal** (Uncertainty-Weighted Mean):

$$\text{weighted} = \frac{\sum_{i=1}^k w_i p_i}{\sum_{i=1}^k w_i}$$

Both signals are logged and saved for reporting purposes.

However, by default, the **fastDTW** alignment algorithm ingests the `weighted_signal`. This design choice ensures that segments with high uncertainty have a downweighted impact on the final alignment.

In [ ]:
"""Aggregates chunk-level author predictions into smoothed regular and uncertainty-weighted style signals.

    Processes rolling prediction frames to smooth out prediction spikes and down-weight high-variance segments.

    Attributes:
        batch_factor: Slicing window factor used to downsample and pool sequential observations.
        build_signals: Aggregates arrays into regular (simple mean) and weighted (inverse-uncertainty mean) style vectors.
        weighted_value: Signal computed using the equation $$\sum (w \cdot p) / \sum w$$, ensuring stable style estimation.
    """
class StyleSignalBuilder:

    def __init__(self, batch_factor: int):
        self.batch_factor = batch_factor

    def build_signals(
        self,
        raw_prediction_map: Dict[str, np.ndarray],
        uncertainty_map: Dict[str, np.ndarray],
        weight_map: Dict[str, np.ndarray],
    ):
        regular_signal_map = {}
        weighted_signal_map = {}
        chunk_uncertainty_map = {}
        style_rows = []

        for file_name, predictions in raw_prediction_map.items():
            uncertainties = uncertainty_map[file_name]
            weights = weight_map[file_name]

            regular_values = []
            weighted_values = []
            chunk_uncertainties = []

            n_full = len(predictions) // self.batch_factor
            for chunk_idx in range(n_full):
                start = chunk_idx * self.batch_factor
                end = start + self.batch_factor

                p = predictions[start:end]
                u = uncertainties[start:end]
                w = weights[start:end]

                regular_value = float(np.mean(p))

                denom = float(np.sum(w))
                if denom <= 1e-12:
                    weighted_value = regular_value
                else:
                    weighted_value = float(np.sum(w * p) / denom)

                avg_uncertainty = float(np.mean(u))

                regular_values.append(regular_value)
                weighted_values.append(weighted_value)
                chunk_uncertainties.append(avg_uncertainty)

                style_rows.append({
                    "file_name": file_name,
                    "signal_type": "regular",
                    "chunk_index": chunk_idx,
                    "value": regular_value,
                    "avg_uncertainty": avg_uncertainty,
                })
                style_rows.append({
                    "file_name": file_name,
                    "signal_type": "weighted",
                    "chunk_index": chunk_idx,
                    "value": weighted_value,
                    "avg_uncertainty": avg_uncertainty,
                })

            regular_signal_map[file_name] = np.asarray(regular_values, dtype=float)
            weighted_signal_map[file_name] = np.asarray(weighted_values, dtype=float)
            chunk_uncertainty_map[file_name] = np.asarray(chunk_uncertainties, dtype=float)

        return regular_signal_map, weighted_signal_map, chunk_uncertainty_map, style_rows

In [ ]:
"""Computes pairwise time-warp distance matrices between style signals using FastDTW.

    Attributes:
        build_distance_matrix: Iterates over text signals to construct a symmetric pairwise distance matrix.
        valid_items: Dictionary filtering out empty or single-token sequences (requires length >= 2).
        valid_names: Sorted or indexed lookup of keys representing texts inside the computed matrix.
    """
class DistanceAnalyzer:

    def build_distance_matrix(self, signal_map: Dict[str, np.ndarray]):
        valid_items = {
            name: np.asarray(signal, dtype=float)
            for name, signal in signal_map.items()
            if signal is not None and len(signal) >= 2
        }
        valid_names = list(valid_items.keys())
        n = len(valid_names)

        if n == 0:
            raise ValueError("No valid style signals with length >= 2")

        distance_matrix = np.zeros((n, n), dtype=float)

        for i in tqdm(range(n), desc="Building fastDTW distance matrix"):
            for j in range(i + 1, n):
                name_i = valid_names[i]
                name_j = valid_names[j]
                distance_value, _ = fastdtw(valid_items[name_i], valid_items[name_j])
                distance_matrix[i, j] = distance_value
                distance_matrix[j, i] = distance_value

        return valid_items, valid_names, distance_matrix

In [ ]:

"""Wrapper adapter for the FasterPAM K-Medoids partition algorithm.

    Handles fallback exception logic dynamically across package updates
    for optional parameter arguments like 'random_state'.

    Attributes:
        n_clusters: The target number of distinct cluster nodes to form.
        metric: Distance measurement approach; strictly requires 'precomputed'.
        labels_: Array containing integer assignment labels mapped per instance post-fitting.
        medoid_indices_: Array tracking computed reference indices selected as cluster medoids.
    """
class KMedoidsAdapter:

    def __init__(self, n_clusters: int = 2, metric: str = "precomputed", max_iter: int = 300, random_state: int = 0):
        self.n_clusters = n_clusters
        self.metric = metric
        self.max_iter = max_iter
        self.random_state = random_state
        self.labels_ = None
        self.medoid_indices_ = None

    def fit(self, distance_matrix: np.ndarray):
        if self.metric != "precomputed":
            raise ValueError("This pipeline expects metric='precomputed'")

        distance_matrix = np.asarray(distance_matrix, dtype=float)
        try:
            result = km.fasterpam(
                diss=distance_matrix,
                medoids=self.n_clusters,
                max_iter=self.max_iter,
                random_state=self.random_state,
            )
        except TypeError:
            result = km.fasterpam(
                diss=distance_matrix,
                medoids=self.n_clusters,
                max_iter=self.max_iter,
            )

        self.labels_ = np.asarray(result.labels)
        self.medoid_indices_ = np.asarray(result.medoids)
        return self

"""Applies statistical anomaly scaling and cluster segregation over computed distance spaces.

    Extracts implicit distribution trends from matrices using unsupervised ensemble trees
    and distance density normalization.

    Attributes:
        config: ExperimentConfig dependency managing clustering boundaries and random seeds.
        analyze: Fits Isolation Forests for anomaly score mining and extracts cluster topologies.
        normalized_sums: Normalized density vectors computed via the row-wise sum ratio formula.
    """
class AnomalyAndClusteringAnalyzer:
    def __init__(self, config: ExperimentConfig):
        self.config = config

    def analyze(self, distance_matrix: np.ndarray):
        distance_matrix = np.asarray(distance_matrix, dtype=float)

        isolation = IsolationForest(
            n_estimators=1000,
            warm_start=True,
            random_state=self.config.random_state,
            contamination="auto",
        )
        isolation.fit(distance_matrix)
        anomaly_predictions = isolation.predict(distance_matrix)
        anomaly_scores = isolation.decision_function(distance_matrix)

        row_sums = distance_matrix.sum(axis=1)
        denom = float(row_sums.sum())
        if denom <= 1e-12:
            normalized_sums = np.zeros_like(row_sums, dtype=float)
        else:
            normalized_sums = row_sums / denom

        n_items = distance_matrix.shape[0]
        n_clusters = min(self.config.kmedoids_clusters, n_items)
        if n_clusters <= 1:
            kmedoids_labels = np.zeros(n_items, dtype=int)
        else:
            kmedoids_model = KMedoidsAdapter(
                n_clusters=n_clusters,
                metric="precomputed",
                random_state=self.config.random_state,
            )
            kmedoids_model.fit(distance_matrix)
            kmedoids_labels = kmedoids_model.labels_

        return {
            "anomaly_predictions": anomaly_predictions,
            "anomaly_scores": anomaly_scores,
            "row_sums": row_sums,
            "normalized_sums": normalized_sums,
            "kmedoids_labels": kmedoids_labels,
        }

In [ ]:
"""Handles directory persistence and structured export of analytics data, metrics, and plots.

    Automates the generation and localized storage of CSV dataframes, dynamic matrix checkpoints,
    and system iteration configurations.

    Attributes:
        paths: PipelinePaths instance establishing directory creation structures upon instantiation.
        save_training_plots: Exports PyTorch classification diagnostic curves (loss and accuracy dimensions) into target images.
        save_reports: Aggregates runtime statistics, saving 7 separate decoupled logging assets (CSV, NPY, JSON).
    """

class ResultWriter:
    def __init__(self, paths: PipelinePaths):
        self.paths = paths
        self.paths.results_root.mkdir(parents=True, exist_ok=True)

    def save_training_plots(self, history: Dict[str, List[float]], iteration: int):
        plt.figure(figsize=(8, 5))
        plt.plot(history.get("train_loss", []), label="train_loss")
        plt.plot(history.get("val_loss", []), label="val_loss")
        plt.title(f"Loss - iteration {iteration}")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(self.paths.results_root / f"loss_{iteration}.png", dpi=150)
        plt.close()

        # Accuracy
        plt.figure(figsize=(8, 5))
        plt.plot(history.get("train_accuracy", []), label="train_accuracy")
        plt.plot(history.get("val_accuracy", []), label="val_accuracy")
        plt.title(f"Accuracy - iteration {iteration}")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(self.paths.results_root / f"accuracy_{iteration}.png", dpi=150)
        plt.close()

    def save_reports(
        self,
        iteration: int,
        valid_names: List[str],
        analysis_results: Dict[str, np.ndarray],
        distance_matrix: np.ndarray,
        prediction_rows: List[Dict[str, Any]],
        style_rows: List[Dict[str, Any]],
        metadata: Dict[str, Any],
    ):
        model_tag = "bert_sen2pro"

        # 1. res_report - anomaly scores
        df_forest = pd.DataFrame({
            "Names": valid_names,
            f"iteration_{iteration}_anomaly_score": analysis_results["anomaly_scores"],
            f"iteration_{iteration}_anomaly_prediction": analysis_results["anomaly_predictions"],
        })
        df_forest.to_csv(self.paths.results_root / f"res_report_{iteration}_{model_tag}.csv", index=False, encoding="utf-8-sig")

        # 2. res_report_summ - normalized summary scores
        df_summ = pd.DataFrame({
            "Names": valid_names,
            f"iteration_{iteration}_row_sum": analysis_results["row_sums"],
            f"iteration_{iteration}_normalized_summary_score": analysis_results["normalized_sums"],
        })
        df_summ.to_csv(self.paths.results_root / f"res_report_summ_{iteration}_{model_tag}.csv", index=False, encoding="utf-8-sig")

        # 3. res_report_Dist - kmedoids labels
        df_dist_labels = pd.DataFrame({
            "Names": valid_names,
            f"iteration_{iteration}_kmedoids_label": analysis_results["kmedoids_labels"],
        })
        df_dist_labels.to_csv(self.paths.results_root / f"res_report_Dist_{iteration}_{model_tag}.csv", index=False, encoding="utf-8-sig")

        # 4. distance matrix npy/csv
        np.save(self.paths.results_root / f"dist_mat_{iteration}.npy", distance_matrix)
        df_mat = pd.DataFrame(distance_matrix, index=valid_names, columns=valid_names)
        df_mat.to_csv(self.paths.results_root / f"dist_mat_{iteration}.csv", encoding="utf-8-sig")

        # 5. predictions segment level
        pd.DataFrame(prediction_rows).to_csv(
            self.paths.results_root / f"predictions_{iteration}.csv",
            index=False,
            encoding="utf-8-sig",
        )

        # 6. style signals
        pd.DataFrame(style_rows).to_csv(
            self.paths.results_root / f"style_signals_{iteration}.csv",
            index=False,
            encoding="utf-8-sig",
        )

        # 7. metadata
        with open(self.paths.results_root / f"metadata_{iteration}.json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

In [ ]:

"""Orchestrates the heBERT transformer-based pipeline for cross-author style analysis.

    Manages dataset preprocessing, embedded caching, Monte Carlo sequence evaluations,
    and fastDTW metrics over dynamic author-imposter pairs with checkpoint logging.

    Attributes:
        paths & config: Core environmental and hyperparameter definitions.
        state & repository: Internal execution logs and data source access abstractions.
        prepare_cached_torah: Normalizes, tokenizes, and segments Torah chapters once per notebook run.
        run_single_pair: Drives the complete cycle for an individual pair, tracking prediction uncertainties.
        run: Grid-searches through all available cross-split author combinations with progress persistence.
    """
class DeepImpostorsBertSen2ProPipeline:

    def __init__(self, paths: PipelinePaths, config: ExperimentConfig):
        self.paths = paths
        self.config = config
        self.state = PipelineState()

        set_global_seed(config.random_state)

        self.repository = CorpusRepository(paths, self.state)
        self.preprocessor = TextPreprocessor()
        self.segmenter = Segmenter(config.segment_word_count)
        self.encoder = Sen2ProBertEncoder(config=config, pooling="mean")
        self.trainer = Trainer(config)
        self.style_builder = StyleSignalBuilder(config.batch_factor)
        self.distance_analyzer = DistanceAnalyzer()
        self.anomaly_analyzer = AnomalyAndClusteringAnalyzer(config)
        self.result_writer = ResultWriter(paths)

        self.cached_torah_text_map = None
        self.cached_torah_tokens_by_file = None
        self.cached_torah_segments_by_file = None

    def validate_environment(self, name_a: Optional[str] = None, name_b: Optional[str] = None):
        print("Running preflight checks...")

        required_dirs = {
            "project_root": self.paths.project_root,
            "torah_dir": self.paths.torah_dir,
            "imposters_dir": self.paths.imposters_dir,
            "imposters1_dir": self.paths.imposters1_dir,
            "results_root": self.paths.results_root,
        }

        for label, path in required_dirs.items():
            if not path.exists():
                raise FileNotFoundError(f"[Preflight] Missing required path: {label} -> {path}")
            if label != "results_root" and not path.is_dir():
                raise NotADirectoryError(f"[Preflight] Expected directory for {label}, got: {path}")

        if name_a is not None:
            path_a = self.paths.imposters_dir / name_a
            if not path_a.exists():
                raise FileNotFoundError(f"[Preflight] Missing Author A folder: {path_a}")
        if name_b is not None:
            path_b = self.paths.imposters1_dir / name_b
            if not path_b.exists():
                raise FileNotFoundError(f"[Preflight] Missing Author B folder: {path_b}")

        torah_files = list(self.paths.torah_dir.glob("*.txt"))
        if len(torah_files) == 0:
            raise FileNotFoundError(f"[Preflight] No .txt files found in {self.paths.torah_dir}")

        print("Preflight OK")

    def load_progress(self) -> Dict[str, Any]:

        if not self.paths.progress_file.exists():
            return {"next_iteration": 1, "completed_pairs": []}
        with open(self.paths.progress_file, "r", encoding="utf-8") as f:
            return json.load(f)

    def save_progress(self, next_iteration: int, completed_pairs: List[Tuple[str, str]]):

        data = {
            "next_iteration": next_iteration,
            "completed_pairs": completed_pairs,
            "updated_at": datetime.now().isoformat(),
        }
        with open(self.paths.progress_file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def reset_progress(self):

        self.state.iteration_number = 1
        self.state.completed_pairs = []
        self.save_progress(next_iteration=1, completed_pairs=[])
        print(f"Progress reset: {self.paths.progress_file}")

    def prepare_cached_torah(self):
        if self.cached_torah_segments_by_file is not None:
            return self.cached_torah_segments_by_file

        print("[Stage] Loading Torah chapters")
        torah_text_map = self.repository.load_torah_corpus()

        print("[Stage] Preprocessing Torah chapters")
        torah_tokens_by_file = {}
        for name, text in torah_text_map.items():
            torah_tokens_by_file[name] = self.preprocessor.preprocess_text(text)

        print("[Stage] Segmenting Torah chapters")
        torah_segments_by_file = {
            name: self.segmenter.tokens_to_segments(tokens)
            for name, tokens in torah_tokens_by_file.items()
        }

        self.cached_torah_text_map = torah_text_map
        self.cached_torah_tokens_by_file = torah_tokens_by_file
        self.cached_torah_segments_by_file = torah_segments_by_file

        print(f"Torah files loaded: {len(torah_segments_by_file)}")
        return torah_segments_by_file

    def run_single_pair(self, author_a: str, author_b: str, iteration: Optional[int] = None):
        iteration = iteration or self.state.iteration_number
        pair_id = f"{iteration}_{author_a}_VS_{author_b}"

        print("=" * 100)
        print(f"Iteration {iteration}: {author_a}  VS  {author_b}")
        print("Sen2Pro-style note: MC Dropout model uncertainty only. No data augmentation.")
        print("Style signal direction: every prediction is P(class 1) = P(Author B).")
        print("fastDTW signal: weighted_signal by default.")
        print("=" * 100)

        self.validate_environment(author_a, author_b)
        torah_segments_by_file = self.prepare_cached_torah()

        print("[Stage] Loading imposters")
        a_texts = self.repository.load_imposter_group_by_name(author_a, "imposters")
        b_texts = self.repository.load_imposter_group_by_name(author_b, "imposters1")

        print("[Stage] Preprocessing imposters")
        a_tokens = self.preprocessor.preprocess_collection(a_texts)
        b_tokens = self.preprocessor.preprocess_collection(b_texts)

        print("[Stage] Segmenting imposters")
        a_segments = self.segmenter.collection_to_segments(a_tokens)
        b_segments = self.segmenter.collection_to_segments(b_tokens)
        print(f"Author A segments: {len(a_segments)}")
        print(f"Author B segments: {len(b_segments)}")

        if len(a_segments) == 0 or len(b_segments) == 0:
            raise ValueError(f"Not enough segments for pair: {author_a}, {author_b}")

        print("[Stage] Encoding Author A with Hebrew BERT + MC Dropout")
        a_repr, a_unc = self.encoder.encode_segments(a_segments, batch_size=self.config.batch_size)

        print("[Stage] Encoding Author B with Hebrew BERT + MC Dropout")
        b_repr, b_unc = self.encoder.encode_segments(b_segments, batch_size=self.config.batch_size)

        print("[Stage] Building balanced training set")
        X, y = build_balanced_training_pairs(
            a_repr,
            b_repr,
            random_state=self.config.random_state + iteration,
        )
        print(f"X shape: {tuple(X.shape)}")
        print(f"y shape: {tuple(y.shape)}")

        print("[Stage] Training classifier")
        classifier, history = self.trainer.train_classifier(X, y, input_dim=self.encoder.input_dim)
        self.result_writer.save_training_plots(history, iteration)

        print("[Stage] Predicting Torah segments")
        predictor = TorahPredictor(self.config, self.encoder)
        raw_prediction_map, uncertainty_map, weight_map, prediction_rows = predictor.predict_torah_segments(
            classifier,
            torah_segments_by_file,
        )

        for row in prediction_rows:
            row["pair_id"] = pair_id
            row["author_A"] = author_a
            row["author_B"] = author_b

        print("[Stage] Building regular and weighted style signals")
        regular_signal_map, weighted_signal_map, chunk_uncertainty_map, style_rows = self.style_builder.build_signals(
            raw_prediction_map,
            uncertainty_map,
            weight_map,
        )

        for row in style_rows:
            row["pair_id"] = pair_id
            row["author_A"] = author_a
            row["author_B"] = author_b

        if self.config.signal_for_distance != "weighted":
            print("Warning: config.signal_for_distance is not 'weighted'. Using regular signal.")
            signal_map_for_distance = regular_signal_map
        else:
            signal_map_for_distance = weighted_signal_map

        print("[Stage] fastDTW distance matrix over style signals")
        valid_items, valid_names, distance_matrix = self.distance_analyzer.build_distance_matrix(signal_map_for_distance)

        print("[Stage] Isolation Forest + K-Medoids")
        analysis_results = self.anomaly_analyzer.analyze(distance_matrix)

        metadata = {
            "iteration": iteration,
            "pair_id": pair_id,
            "author_A": author_a,
            "author_B": author_b,
            "bert_model_name": self.config.bert_model_name,
            "sen2pro_style": "model uncertainty via MC Dropout only; no data augmentation",
            "mc_samples": self.config.mc_samples,
            "segment_word_count": self.config.segment_word_count,
            "batch_factor": self.config.batch_factor,
            "max_length": self.config.max_length,
            "freeze_bert": self.config.freeze_bert,
            "hidden_size": self.encoder.hidden_size,
            "input_dim": self.encoder.input_dim,
            "classifier": "Linear(input_dim -> 256) + ReLU + Dropout + Linear(256 -> 2)",
            "prediction_used_for_style_signal": "P(class 1) = P(Author B)",
            "regular_signal_saved": True,
            "weighted_signal_saved": True,
            "signal_used_for_fastdtw": self.config.signal_for_distance,
            "distance_method": "fastDTW over style signals",
            "created_at": datetime.now().isoformat(),
        }

        print("[Stage] Saving reports")
        self.result_writer.save_reports(
            iteration=iteration,
            valid_names=valid_names,
            analysis_results=analysis_results,
            distance_matrix=distance_matrix,
            prediction_rows=prediction_rows,
            style_rows=style_rows,
            metadata=metadata,
        )

        self._cleanup_iteration_objects(
            a_repr, b_repr, a_unc, b_unc, X, y, classifier,
            raw_prediction_map, uncertainty_map, weight_map,
            regular_signal_map, weighted_signal_map, chunk_uncertainty_map,
            distance_matrix,
        )

        print(f"Iteration {iteration} finished successfully")
        return {
            "iteration": iteration,
            "pair_id": pair_id,
            "valid_names": valid_names,
            "analysis_results": analysis_results,
        }

    def run(self, resume: bool = False):
        self.validate_environment()

        authors_a = self.repository.list_author_names_from_imposters()
        authors_b = self.repository.list_author_names_from_imposters1()
        all_pairs = [(a, b) for a in authors_a for b in authors_b]

        if resume:
            progress = self.load_progress()
            next_iteration = int(progress.get("next_iteration", 1))
            completed_pairs = [tuple(x) for x in progress.get("completed_pairs", [])]
            print(f"Resuming from iteration {next_iteration}")
        else:
            next_iteration = 1
            completed_pairs = []
            self.save_progress(next_iteration=1, completed_pairs=[])

        self.state.completed_pairs = completed_pairs

        for pair_index, (author_a, author_b) in enumerate(all_pairs, start=1):
            if (author_a, author_b) in self.state.completed_pairs:
                print(f"Skipping completed pair: {author_a} VS {author_b}")
                continue

            iteration = next_iteration
            self.state.iteration_number = iteration

            try:
                self.run_single_pair(author_a, author_b, iteration=iteration)
                self.state.completed_pairs.append((author_a, author_b))
                next_iteration += 1
                self.save_progress(next_iteration=next_iteration, completed_pairs=self.state.completed_pairs)
            except Exception as e:
                print("ERROR during iteration")
                print(f"Pair: {author_a} VS {author_b}")
                traceback.print_exc()
                self.save_progress(next_iteration=iteration, completed_pairs=self.state.completed_pairs)
                raise e

        print("All pairs finished")

    def _cleanup_iteration_objects(self, *objects):
        for obj in objects:
            del obj
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


**Full Execution Cell**



Note: If the program crashed and you need to resume from the last checkpoint, run the cell below instead.

In [ ]:
"""
Main Transformer Pipeline Execution Script.

Initializes the workspace paths and hyperparameter overrides for the heBERT + Sen2Pro
uncertainty framework, resets any previous iteration checkpoints, and starts a fresh,
end-to-end grid-search run across all author pairings.
"""

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_bert_sen2pro_results/"),
)

config = ExperimentConfig(
    bert_model_name="avichr/heBERT",
    max_length=128,
    segment_word_count=50,
    batch_factor=2,
    mc_samples=15,
    epochs=12,
    batch_size=8,
    learning_rate=2e-4,
    dropout_rate=0.2,
    freeze_bert=True,
    kmedoids_clusters=2,
    signal_for_distance="weighted",
    early_stopping_patience=3,
)

pipeline = DeepImpostorsBertSen2ProPipeline(paths, config)
pipeline.reset_progress()
pipeline.run(resume=False)

# ============================================================
# SANITY EXTENSION: Recovery Crash / Resume Test Pipeline
# ============================================================

In [ ]:

class DeepImpostorsBertSen2ProRecoverySanityPipeline(DeepImpostorsBertSen2ProPipeline):
    """
    Recovery-sanity extension for the heBERT + Sen2Pro pipeline.

    This class is intended only for controlled sanity checks around the
    progress/recovery mechanism. It uses the same `run_single_pair()` logic
    as the regular pipeline, but limits the run to a small number of author
    pairs and can intentionally crash at a selected iteration.

    The test verifies that:
        - progress.json stores `next_iteration` correctly.
        - completed pairs are saved only after successful iterations.
        - a failed iteration is not marked as completed.
        - resume continues from the next unfinished pair.

    Use the regular DeepImpostorsBertSen2ProPipeline for real experiments.
    """

    def _get_sanity_pairs(self, max_iterations: int) -> List[Tuple[str, str]]:
        authors_a = self.repository.list_author_names_from_imposters()
        authors_b = self.repository.list_author_names_from_imposters1()
        all_pairs = [(a, b) for a in authors_a for b in authors_b]

        if not all_pairs:
            raise ValueError("No author pairs found for recovery sanity check.")

        return all_pairs[:min(len(all_pairs), max_iterations)]

    def run_recovery_sanity_check(
        self,
        max_iterations: int = 5,
        crash_at_iteration: int = 3,
        reset_before_run: bool = True,
    ):
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")
        if crash_at_iteration < 1:
            raise ValueError("crash_at_iteration must be at least 1.")
        if crash_at_iteration > max_iterations:
            raise ValueError("crash_at_iteration cannot be greater than max_iterations.")

        self.validate_environment()
        sanity_pairs = self._get_sanity_pairs(max_iterations)

        if crash_at_iteration > len(sanity_pairs):
            raise ValueError(
                f"crash_at_iteration={crash_at_iteration} is larger than the available "
                f"sanity pairs count={len(sanity_pairs)}."
            )

        if reset_before_run:
            self.reset_progress()

        progress = self.load_progress()
        next_iteration = int(progress.get("next_iteration", 1))
        completed_pairs = [tuple(pair) for pair in progress.get("completed_pairs", [])]
        self.state.completed_pairs = completed_pairs

        print("=" * 100)
        print("RECOVERY SANITY CHECK - PHASE 1")
        print(f"Running up to {len(sanity_pairs)} sanity iterations")
        print(f"Intentional crash at iteration {crash_at_iteration}")
        print("=" * 100)

        for author_a, author_b in sanity_pairs:
            pair = (author_a, author_b)

            if pair in self.state.completed_pairs:
                print(f"Skipping completed sanity pair: {author_a} VS {author_b}")
                continue

            iteration = next_iteration
            self.state.iteration_number = iteration

            try:
                if iteration == crash_at_iteration:
                    raise RuntimeError(
                        f"Intentional crash for recovery sanity check at iteration {iteration}"
                    )

                self.run_single_pair(author_a, author_b, iteration=iteration)
                self.state.completed_pairs.append(pair)
                next_iteration += 1

                self.save_progress(
                    next_iteration=next_iteration,
                    completed_pairs=self.state.completed_pairs,
                )

            except Exception as e:
                print("SANITY CHECK CRASH / ERROR")
                print(f"Pair: {author_a} VS {author_b}")
                print(f"Saving progress with next_iteration={iteration}")

                self.save_progress(
                    next_iteration=iteration,
                    completed_pairs=self.state.completed_pairs,
                )

                traceback.print_exc()
                raise e

        print("Recovery sanity phase 1 finished without reaching the intentional crash.")

    def resume_from_progress_for_sanity(self, max_iterations: int = 5):
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")

        self.validate_environment()
        sanity_pairs = self._get_sanity_pairs(max_iterations)

        progress = self.load_progress()
        next_iteration = int(progress.get("next_iteration", 1))
        completed_pairs = [tuple(pair) for pair in progress.get("completed_pairs", [])]
        self.state.completed_pairs = completed_pairs

        print("=" * 100)
        print("RECOVERY SANITY CHECK - PHASE 2")
        print(f"Resuming from progress.json with next_iteration={next_iteration}")
        print(f"Sanity target iterations: {len(sanity_pairs)}")
        print("=" * 100)

        for author_a, author_b in sanity_pairs:
            pair = (author_a, author_b)

            if pair in self.state.completed_pairs:
                print(f"Skipping completed sanity pair: {author_a} VS {author_b}")
                continue

            iteration = next_iteration
            self.state.iteration_number = iteration

            try:
                self.run_single_pair(author_a, author_b, iteration=iteration)
                self.state.completed_pairs.append(pair)
                next_iteration += 1

                self.save_progress(
                    next_iteration=next_iteration,
                    completed_pairs=self.state.completed_pairs,
                )

            except Exception as e:
                print("ERROR during recovery sanity resume")
                print(f"Pair: {author_a} VS {author_b}")
                print(f"Saving progress with next_iteration={iteration}")

                self.save_progress(
                    next_iteration=iteration,
                    completed_pairs=self.state.completed_pairs,
                )

                traceback.print_exc()
                raise e

        print("Recovery sanity phase 2 finished successfully.")
        print(f"Final next_iteration={next_iteration}")
        print(f"Completed sanity pairs={len(self.state.completed_pairs)}")

============================================================


Cell 18 — Crash Recovery


============================================================


In the event of a Google Colab session crash mid-execution:
1. Re-run all preceding cells that define classes and utilities.
2. Execute this recovery cell.
#
# The pipeline will automatically parse 'progress.json' and
# resume execution from the next incomplete pair.

In [ ]:

"""
Transformer Pipeline Recovery and Continuation Script.

Loads existing progress states from the JSON tracking file to resume the heBERT
and Monte Carlo dropout execution loop exactly from the last uncompleted
author-imposter pairing, preventing redundant grid-search calculations.
"""
paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_bert_sen2pro_results/"),
)

config = ExperimentConfig(
    bert_model_name="avichr/heBERT",
    max_length=128,
    segment_word_count=50,
    batch_factor=2,
    mc_samples=15,
    epochs=12,
    batch_size=8,
    learning_rate=2e-4,
    dropout_rate=0.2,
    freeze_bert=True,
    kmedoids_clusters=2,
    signal_for_distance="weighted",
    early_stopping_patience=3,
)

pipeline = DeepImpostorsBertSen2ProPipeline(paths, config)
pipeline.run(resume=True)

# ============================================================
# RECOVERY SANITY CHECK - PHASE 1: Intentional Crash
# ============================================================

In [ ]:

"""
This sanity cell intentionally crashes the pipeline at a controlled iteration.

Expected behavior:
    - Iteration 1 finishes successfully.
    - Iteration 2 finishes successfully.
    - Iteration 3 crashes intentionally.
    - progress.json should contain next_iteration == 3.
    - completed_pairs should contain exactly the first 2 completed pairs.

This proves that the failed iteration is not marked as completed.
"""

SANITY_MAX_ITERATIONS = 5
SANITY_CRASH_AT_ITERATION = 3

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_bert_sen2pro_recovery_sanity_check/"),
)

config = ExperimentConfig(
    bert_model_name="avichr/heBERT",
    max_length=128,
    segment_word_count=50,
    batch_factor=2,
    mc_samples=3,          # Faster sanity check. Full run uses 15.
    epochs=1,              # Faster sanity check. Full run uses 12.
    batch_size=8,
    learning_rate=2e-4,
    dropout_rate=0.2,
    freeze_bert=True,
    kmedoids_clusters=2,
    signal_for_distance="weighted",
    early_stopping_patience=1,
)

pipeline = DeepImpostorsBertSen2ProRecoverySanityPipeline(paths, config)

try:
    pipeline.run_recovery_sanity_check(
        max_iterations=SANITY_MAX_ITERATIONS,
        crash_at_iteration=SANITY_CRASH_AT_ITERATION,
        reset_before_run=True,
    )

except RuntimeError as e:
    if "Intentional crash for recovery sanity check" not in str(e):
        raise

    print("\nCaught expected intentional crash.")
    print("This is part of the recovery sanity check, not a real failure.")

progress = pipeline.load_progress()

print("\n" + "=" * 100)
print("PROGRESS AFTER INTENTIONAL CRASH")
print("=" * 100)
print(progress)

assert int(progress["next_iteration"]) == SANITY_CRASH_AT_ITERATION, (
    f"Expected next_iteration == {SANITY_CRASH_AT_ITERATION}, "
    f"got {progress['next_iteration']}"
)

assert len(progress["completed_pairs"]) == SANITY_CRASH_AT_ITERATION - 1, (
    f"Expected {SANITY_CRASH_AT_ITERATION - 1} completed pairs, "
    f"got {len(progress['completed_pairs'])}"
)

print("\nPhase 1 passed.")
print("The failed iteration was not marked as completed.")

In [ ]:
# ============================================================
# RECOVERY SANITY CHECK - PHASE 2: Resume After Crash
# ============================================================

"""
This sanity cell resumes the same limited run after the intentional crash.

Expected behavior:
    - The pipeline reads progress.json.
    - It resumes from next_iteration == 3.
    - It skips already completed pairs.
    - It completes iterations 3, 4, and 5.
    - progress.json should end with next_iteration == 6.
"""

SANITY_MAX_ITERATIONS = 5

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_bert_sen2pro_recovery_sanity_check/"),
)

config = ExperimentConfig(
    bert_model_name="avichr/heBERT",
    max_length=128,
    segment_word_count=50,
    batch_factor=2,
    mc_samples=3,
    epochs=1,
    batch_size=8,
    learning_rate=2e-4,
    dropout_rate=0.2,
    freeze_bert=True,
    kmedoids_clusters=2,
    signal_for_distance="weighted",
    early_stopping_patience=1,
)

pipeline = DeepImpostorsBertSen2ProRecoverySanityPipeline(paths, config)

pipeline.resume_from_progress_for_sanity(
    max_iterations=SANITY_MAX_ITERATIONS,
)

progress = pipeline.load_progress()

print("\n" + "=" * 100)
print("PROGRESS AFTER RESUME")
print("=" * 100)
print(progress)

assert int(progress["next_iteration"]) == SANITY_MAX_ITERATIONS + 1, (
    f"Expected next_iteration == {SANITY_MAX_ITERATIONS + 1}, "
    f"got {progress['next_iteration']}"
)

assert len(progress["completed_pairs"]) == SANITY_MAX_ITERATIONS, (
    f"Expected {SANITY_MAX_ITERATIONS} completed pairs, "
    f"got {len(progress['completed_pairs'])}"
)

print("\nPhase 2 passed.")
print("Recovery sanity check completed successfully.")

## Single Pair Sanity Check

> **Recommended:** Execute this sanity check on a single pair before initiating a full pipeline run.

### Key Details & Training Logic:
* **Epochs:** Configured to run for `epochs=12` to match the original pipeline specifications.
* **Model Selection:** The `Trainer` automatically selects the **best-performing checkpoint** based on the lowest validation loss.
* **Note:** The final evaluated model is not guaranteed to be from the last epoch, but rather the most optimal one.

In [ ]:

"""
Single-Pair Transformer Pipeline Sanity Check.

Executes an isolated validation loop comparing a specific author pairing
(Alharizi vs. Berechiah) using the heBERT + MC Dropout architecture, confirming
the integrity of tokenization, feature embedding, model training, and output report generation.
"""

IMPOSTER_A_NAME = "יהודה אלחריזי"
IMPOSTER_B_NAME = "ברכיה הנקדן"

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_bert_sen2pro_single_pair_sanity_check/"),
)

config = ExperimentConfig(
    bert_model_name="avichr/heBERT",
    max_length=128,
    segment_word_count=50,
    batch_factor=2,
    mc_samples=15,
    epochs=12,
    batch_size=8,
    learning_rate=2e-4,
    dropout_rate=0.2,
    freeze_bert=True,
    kmedoids_clusters=2,
    signal_for_distance="weighted",
    early_stopping_patience=3,
)

pipeline = DeepImpostorsBertSen2ProPipeline(paths, config)

print("=" * 100)
print("SINGLE PAIR SANITY CHECK")
print(f"Author A: {IMPOSTER_A_NAME}")
print(f"Author B: {IMPOSTER_B_NAME}")
print("Prediction direction: P(class 1) = P(Author B)")
print("Sen2Pro-style: MC Dropout uncertainty only; no data augmentation")
print(f"Epochs: {config.epochs}")
print(f"Early stopping patience: {config.early_stopping_patience}")
print("Best model selection: best validation loss")
print("=" * 100)

pipeline.run_single_pair(IMPOSTER_A_NAME, IMPOSTER_B_NAME, iteration=1)

print("\nGenerated files:")
for file_path in sorted(paths.results_root.glob("*")):
    print("-", file_path.name)

## Cell 20 — Debug & Validation

This cell prints data shapes, counts, and dimensions to verify pipeline integrity and ensure everything is functioning as expected.

### Verification Checklist:
* **Dataset Metadata:**
  * Number of biblical source files & chapter names.
  * Text counts per author.
* **Preprocessing & Tokenization:**
  * Token counts post-preprocessing.
  * Number of segments per side.
* **Tensors & Matrices:**
  * **Shape of $X$:** Expected to be `[num_samples, 1536]` (assuming `hidden_size=768` due to mean/variance concatenation).
  * Distance matrix dimensions.
* **Signals & Predictions:**
  * Prediction counts per chapter.
  * Style signal length per chapter.
* **Outputs & Clustering:**
  * `valid_names`
  * Anomaly scores
  * K-Medoids cluster labels

In [ ]:

"""
End-to-End Pipeline Debug and Architecture Dry-Run.

Executes a minimized standalone validation flow using low Monte Carlo iteration loops
and truncated data sequences, logging the tensor shape mutations, layer dimensions,
data loaders, and matrix topologies across each processing pipeline phase.
"""
IMPOSTER_A_NAME = "יהודה אלחריזי"
IMPOSTER_B_NAME = "ברכיה הנקדן"

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_bert_sen2pro_debug/"),
)

config = ExperimentConfig(
    bert_model_name="avichr/heBERT",
    max_length=128,
    segment_word_count=50,
    batch_factor=2,
    mc_samples=5,      # Fast debugging. For full execution, revert to 15.
    epochs=1,          # Fast debugging. For full execution, revert to 3 or 5.
    batch_size=8,
    learning_rate=2e-4,
    dropout_rate=0.2,
    freeze_bert=True,
    kmedoids_clusters=2,
    signal_for_distance="weighted",
)

pipeline = DeepImpostorsBertSen2ProPipeline(paths, config)
pipeline.validate_environment(IMPOSTER_A_NAME, IMPOSTER_B_NAME)

torah_segments_by_file = pipeline.prepare_cached_torah()
torah_names = list(torah_segments_by_file.keys())

print("=" * 100)
print("TORAH")
print("=" * 100)
print("Number of Torah files:", len(torah_names))
print("Torah file names:", torah_names)
print("Torah segments per file:", {name: len(segs) for name, segs in torah_segments_by_file.items()})

print("\n" + "=" * 100)
print("AUTHORS")
print("=" * 100)
a_texts = pipeline.repository.load_imposter_group_by_name(IMPOSTER_A_NAME, "imposters")
b_texts = pipeline.repository.load_imposter_group_by_name(IMPOSTER_B_NAME, "imposters1")
print("Author A text files:", len(a_texts))
print("Author B text files:", len(b_texts))

a_tokens = pipeline.preprocessor.preprocess_collection(a_texts)
b_tokens = pipeline.preprocessor.preprocess_collection(b_texts)
print("Author A token counts:", [len(x) for x in a_tokens])
print("Author B token counts:", [len(x) for x in b_tokens])

a_segments = pipeline.segmenter.collection_to_segments(a_tokens)
b_segments = pipeline.segmenter.collection_to_segments(b_tokens)
print("Author A segments:", len(a_segments))
print("Author B segments:", len(b_segments))

print("\n" + "=" * 100)
print("ENCODING SHAPES")
print("=" * 100)
a_repr, a_unc = pipeline.encoder.encode_segments(a_segments[:min(16, len(a_segments))], batch_size=config.batch_size)
b_repr, b_unc = pipeline.encoder.encode_segments(b_segments[:min(16, len(b_segments))], batch_size=config.batch_size)
X, y = build_balanced_training_pairs(a_repr, b_repr, random_state=config.random_state)
print("hidden_size:", pipeline.encoder.hidden_size)
print("Expected input_dim = hidden_size * 2:", pipeline.encoder.input_dim)
print("X shape:", tuple(X.shape))
print("y shape:", tuple(y.shape))
print("y labels distribution:", dict(zip(*np.unique(y.numpy(), return_counts=True))))

print("\n" + "=" * 100)
print("MINI TRAIN + MINI TORAH PREDICTION")
print("=" * 100)
classifier, history = pipeline.trainer.train_classifier(X, y, input_dim=pipeline.encoder.input_dim)

mini_torah = {name: segs[:min(8, len(segs))] for name, segs in torah_segments_by_file.items()}
predictor = TorahPredictor(config, pipeline.encoder)
raw_prediction_map, uncertainty_map, weight_map, prediction_rows = predictor.predict_torah_segments(classifier, mini_torah)

print("Prediction counts per Torah file:", {k: len(v) for k, v in raw_prediction_map.items()})

regular_signal_map, weighted_signal_map, chunk_uncertainty_map, style_rows = pipeline.style_builder.build_signals(
    raw_prediction_map,
    uncertainty_map,
    weight_map,
)
print("Regular signal length per file:", {k: len(v) for k, v in regular_signal_map.items()})
print("Weighted signal length per file:", {k: len(v) for k, v in weighted_signal_map.items()})

valid_items, valid_names, distance_matrix = pipeline.distance_analyzer.build_distance_matrix(weighted_signal_map)
print("Distance matrix shape:", distance_matrix.shape)
print("Valid names:", valid_names)

analysis_results = pipeline.anomaly_analyzer.analyze(distance_matrix)
print("Anomaly scores:", analysis_results["anomaly_scores"])
print("K-Medoids labels:", analysis_results["kmedoids_labels"])